# Fetching real ZTF light curves for RQ3b

The RQ3 comparison in `lc_classifier_plasticc.ipynb` is currently answered *by proxy*:
PLAsTiCC and ZTF are each classified under a matched protocol, but the two sides go
through **different feature extractors** — PLAsTiCC through the `light-curve` package,
ZTF through ALeRCE's own precomputed feature service. A model therefore cannot be
carried from one domain to the other, so "does a simulation-trained model work on real
data?" is never asked directly.

This notebook removes that obstacle by pulling the **raw per-epoch photometry** for every
gold object, so that §9 of the PLAsTiCC notebook can re-extract ZTF features with the
*same* `light-curve` extractor used on the simulations.

**Read-only contract.** The gold artefacts under `data/gold/` are inputs only. Nothing
here writes to them. All output goes to the new directory `data/rq3b_ztf_lc/`.

**Resumability.** Every object is cached individually on disk and the cache is consulted
before any network call, so re-running this notebook re-fetches nothing already stored.
A run interrupted at any point can simply be restarted.

## 0. Setup

In [1]:
import os, sys, json, time, datetime, warnings
from pathlib import Path
import threading
from concurrent.futures import ThreadPoolExecutor, as_completed

import numpy as np
import pandas as pd
from tqdm import tqdm

warnings.filterwarnings("ignore")

# --- read-only inputs -------------------------------------------------------
GOLD = Path("data/gold")                       # NEVER written to by this notebook

# --- new outputs (rq3b_ prefix; nothing existing is shadowed) ---------------
OUTDIR = Path("data/rq3b_ztf_lc")
CACHE  = OUTDIR / "_cache_detections"          # one JSON per object, resumable
OUTDIR.mkdir(parents=True, exist_ok=True)
CACHE.mkdir(parents=True, exist_ok=True)

DET_PARQUET = OUTDIR / "rq3b_ztf_detections.parquet"
MANIFEST    = OUTDIR / "rq3b_ztf_detections_manifest.json"

# --- networking: the pattern already proven in build_dataset.ipynb ----------
WORKERS       = 6        # concurrent ALeRCE fetches (be polite)
MAX_RETRIES   = 4
RETRY_BACKOFF = 2.0      # seconds; exponential -> 2, 4, 8

# ALeRCE rate-limits per IP and answers HTTP 429 once the quota is exceeded. An
# unpaced six-thread pool trips that after a few thousand objects, after which every
# request fails and the run makes no further progress. Requests are therefore paced
# globally, and the pace adapts downwards whenever a 429 comes back. See section 3.
TARGET_RATE_PER_MIN = 90.0   # global ceiling summed across all workers
MIN_RATE_PER_MIN    = 15.0   # floor that the adaptive backoff will not go below
THROTTLE_SLEEP_S    = 60.0   # stand-down after a 429 before retrying
MAX_THROTTLE_WAITS  = 8      # per object, before it is recorded as failed
RUN_DATE      = datetime.date.today().isoformat()

QUICK = os.environ.get("RQ3B_QUICK", "0") == "1"   # tiny smoke run

print(f"output dir : {OUTDIR.resolve()}")
print(f"cache dir  : {CACHE.resolve()}")
print(f"workers={WORKERS}  retries={MAX_RETRIES}  backoff={RETRY_BACKOFF}s  QUICK={QUICK}")

output dir : F:\Poly\Thesis\astro-transient-classification\data\rq3b_ztf_lc
cache dir  : F:\Poly\Thesis\astro-transient-classification\data\rq3b_ztf_lc\_cache_detections
workers=6  retries=4  backoff=2.0s  QUICK=False


## 1. The object list, read straight from the gold artefacts

`gold_labels.parquet` is opened read-only and nothing is written back. The oid list it
carries *is* the population every downstream chapter was built on, so re-deriving it any
other way would risk a silently different sample.

In [2]:
labels = pd.read_parquet(GOLD / "gold_labels.parquet")
GOLD_OIDS = labels["oid"].astype(str).tolist()
GOLD_OID_SET = set(GOLD_OIDS)
assert len(GOLD_OIDS) == len(GOLD_OID_SET), "duplicate oids in gold_labels"

if QUICK:
    GOLD_OIDS = GOLD_OIDS[:60]
    print(f"QUICK mode: restricted to {len(GOLD_OIDS)} objects")

print(f"gold objects to fetch: {len(GOLD_OIDS):,}")
print(f"coarse-class breakdown:\n{labels['coarse'].value_counts().to_string()}")

gold objects to fetch: 11,826
coarse-class breakdown:
coarse
SN     7728
VS     3517
AGN     581


## 2. The ALeRCE client and the exact detections signature

The signature is *printed from the installed package* rather than assumed, because the
`survey=` argument became mandatory in recent client versions and silently defaulting to
ZTF is deprecated.

In [3]:
import inspect
from alerce.core import Alerce
import alerce as _alerce_pkg

alerce = Alerce()

print("alerce version :", getattr(_alerce_pkg, "__version__", "unknown"))
print("query_detections", inspect.signature(alerce.query_detections))

alerce version : 2.3.1
query_detections (oid: str | int, format: str = 'json', survey: str | None = None, index=None, sort=None)


## 3. Per-object fetch, cached and retried

The design mirrors `build_dataset.ipynb`: check disk first, then up to four attempts with
a growing sleep between them.

**Rate limiting.** One thing the `build_dataset.ipynb` pattern does not survive at this
volume: ALeRCE caps requests per IP and returns HTTP 429 once the cap is hit. Six threads
running flat out reach that ceiling after roughly 3,500 objects, and from then on every
request is rejected, so the pool spins without making progress. Requests are therefore
paced by a single global limiter shared by all six workers, and a 429 halves the pace
rather than counting as one of the four error retries. The ceiling measured here sits
above 90 requests/minute, which is the target the limiter aims at.

One deliberate refinement. A response that *succeeds but contains no photometry* is a
real astrophysical fact about that object, so it is cached (as an empty record) and never
re-queried. A response that *raises on all four attempts* is a transport failure, so it is
**not** cached and will be retried on the next run. The manifest reports the two
populations separately — conflating them would let a network outage masquerade as
"objects with no data".

In [4]:
# Columns we commit to retaining. Everything else ALeRCE returns is dropped, both to keep
# the parquet small and to make the contract with §9 explicit.
KEEP_COLS = ["mjd", "fid", "magpsf", "sigmapsf", "isdiffpos"]


def _isdiffpos_sign(v):
    """ALeRCE returns isdiffpos as 't'/'f' or '1'/'-1' depending on vintage.

    Returns +1 for a positive (brightening) difference-image residual, -1 for a negative
    one, and NaN if unparseable. The sign is what carries the direction of the flux
    excursion once magnitudes are converted in §9 of the PLAsTiCC notebook.
    """
    if v is None or (isinstance(v, float) and np.isnan(v)):
        return np.nan
    s = str(v).strip().lower()
    if s in ("t", "true", "1", "+1", "1.0"):
        return 1
    if s in ("f", "false", "-1", "-1.0"):
        return -1
    return np.nan


class RateLimiter:
    """Global, thread-safe pacer with 429-driven adaptive backoff.

    ``acquire`` hands out request slots no faster than the current interval, across
    every worker. ``penalise`` halves the rate after a 429; ``reward`` walks it back
    towards the target after a long clean run, so one transient throttle does not
    permanently cripple the fetch.
    """

    def __init__(self, rate_per_min, min_rate_per_min):
        self._lock = threading.Lock()
        self._base = 60.0 / rate_per_min
        self._max = 60.0 / min_rate_per_min
        self._interval = self._base
        self._next = time.monotonic()
        self._clean = 0

    def acquire(self):
        with self._lock:
            slot = max(time.monotonic(), self._next)
            self._next = slot + self._interval
        delay = slot - time.monotonic()
        if delay > 0:
            time.sleep(delay)

    def penalise(self):
        with self._lock:
            self._interval = min(self._interval * 2.0, self._max)
            self._clean = 0
            return 60.0 / self._interval

    def reward(self):
        with self._lock:
            self._clean += 1
            if self._clean >= 250 and self._interval > self._base:
                self._interval = max(self._base, self._interval / 1.5)
                self._clean = 0


LIMITER = RateLimiter(TARGET_RATE_PER_MIN, MIN_RATE_PER_MIN)
_throttle_events = []          # (oid, new_rate) pairs, summarised in the manifest


def _is_rate_limited(exc) -> bool:
    s = str(exc).lower()
    return "429" in s or "too many requests" in s


def _parse_detections(oid: str, det) -> list:
    det = det.reset_index()                    # oid is often the index in pandas format
    missing = [c for c in KEEP_COLS if c not in det.columns]
    if missing:
        raise KeyError(f"detections for {oid} missing columns {missing}")
    sub = det[KEEP_COLS].copy()
    sub["isdiffpos_sign"] = sub["isdiffpos"].map(_isdiffpos_sign)
    sub["isdiffpos"] = sub["isdiffpos"].astype(str)
    for c in ("mjd", "magpsf", "sigmapsf"):
        sub[c] = pd.to_numeric(sub[c], errors="coerce")
    sub["fid"] = pd.to_numeric(sub["fid"], errors="coerce").astype("Int64")
    return sub.to_dict(orient="records")


def _fetch_one(oid: str):
    """Return (oid, records | None, status).

    status is one of 'cached', 'fetched', 'empty' (server answered, no photometry) or
    'failed'. Only non-'failed' outcomes are written to the cache.

    A 429 is not a failure, it is the server asking us to slow down, so it does not
    consume one of the four error retries: the limiter is slowed, the worker stands
    down for THROTTLE_SLEEP_S, and the same object is tried again (up to
    MAX_THROTTLE_WAITS times).
    """
    cache = CACHE / f"{oid}.json"
    if cache.exists():
        try:
            return oid, json.loads(cache.read_text()), "cached"
        except Exception:
            cache.unlink(missing_ok=True)      # corrupt cache entry -> refetch

    attempts, throttles = 0, 0
    while True:
        LIMITER.acquire()
        try:
            det = alerce.query_detections(oid, format="pandas", survey="ztf")
            LIMITER.reward()

            if det is None or len(det) == 0:
                cache.write_text(json.dumps([]))
                return oid, [], "empty"

            recs = _parse_detections(oid, det)
            cache.write_text(json.dumps(recs, default=str))
            return oid, recs, "fetched"

        except Exception as exc:
            if _is_rate_limited(exc):
                throttles += 1
                if throttles > MAX_THROTTLE_WAITS:
                    return oid, None, "failed"
                new_rate = LIMITER.penalise()
                _throttle_events.append((oid, new_rate))
                time.sleep(THROTTLE_SLEEP_S)
                continue                       # does NOT consume an error retry

            attempts += 1
            if attempts >= MAX_RETRIES:
                return oid, None, "failed"     # deliberately NOT cached
            time.sleep(RETRY_BACKOFF * (2 ** (attempts - 1)))   # 2, 4, 8 s

## 4. Run the pool

Six threads, resumable. Expect roughly an hour on a cold cache; a warm cache completes in
seconds because no request leaves the machine.

In [5]:
t0 = time.perf_counter()
by_oid, status_counts = {}, {"cached": 0, "fetched": 0, "empty": 0, "failed": 0}
failed_oids = []

todo = GOLD_OIDS
n_precached = sum(1 for o in todo if (CACHE / f"{o}.json").exists())
print(f"{n_precached:,} of {len(todo):,} objects already cached; "
      f"{len(todo) - n_precached:,} to fetch\n")

with ThreadPoolExecutor(max_workers=WORKERS) as ex:
    futs = {ex.submit(_fetch_one, o): o for o in todo}
    done = 0
    for f in tqdm(as_completed(futs), total=len(futs), desc="ztf-detections"):
        oid, recs, status = f.result()
        status_counts[status] += 1
        if recs:
            by_oid[oid] = recs
        if status == "failed":
            failed_oids.append(oid)
        done += 1
        if done % 500 == 0 or done == len(futs):
            el = time.perf_counter() - t0
            print(f"  [{done:6,}/{len(futs):,}]  {el/60:5.1f} min elapsed  "
                  f"| with-photometry {len(by_oid):,}  failed {len(failed_oids):,}",
                  flush=True)

ELAPSED_S = time.perf_counter() - t0
print(f"\nfetch finished in {ELAPSED_S/60:.1f} min")
print("status:", status_counts)
if failed_oids:
    print(f"WARNING: {len(failed_oids)} objects failed all {MAX_RETRIES} attempts and "
          f"were NOT cached; re-run this notebook to retry only those.")
    print("first few:", failed_oids[:10])

3,490 of 11,826 objects already cached; 8,336 to fetch




ztf-detections:   0%|          | 0/11826 [00:00<?, ?it/s]


ztf-detections:   1%|          | 127/11826 [00:00<00:09, 1242.58it/s]


ztf-detections:   2%|▏         | 252/11826 [00:00<00:13, 855.04it/s] 


ztf-detections:   3%|▎         | 345/11826 [00:00<00:14, 772.80it/s]

...


ztf-detections: 100%|█████████▉| 11825/11826 [1:32:41<00:00,  1.34it/s]

  [11,826/11,826]   92.7 min elapsed  | with-photometry 11,826  failed 0



ztf-detections: 100%|██████████| 11826/11826 [1:32:42<00:00,  1.41it/s]


ztf-detections: 100%|██████████| 11826/11826 [1:32:42<00:00,  2.13it/s]


fetch finished in 92.7 min
status: {'cached': 3490, 'fetched': 8336, 'empty': 0, 'failed': 0}


## 5. Stack to one parquet, and write the manifest

In [6]:
frames = []
for oid, recs in by_oid.items():
    d = pd.DataFrame(recs)
    d.insert(0, "oid", oid)
    frames.append(d)

det_all = (pd.concat(frames, ignore_index=True) if frames
           else pd.DataFrame(columns=["oid", *KEEP_COLS, "isdiffpos_sign"]))

# Types, and a stable ordering that makes the per-object groupby in §9 cheap.
det_all["oid"] = det_all["oid"].astype(str)
for c in ("mjd", "magpsf", "sigmapsf"):
    det_all[c] = pd.to_numeric(det_all[c], errors="coerce")
det_all["fid"] = pd.to_numeric(det_all["fid"], errors="coerce").astype("Int64")
det_all["isdiffpos_sign"] = pd.to_numeric(det_all["isdiffpos_sign"], errors="coerce")
det_all = det_all.sort_values(["oid", "fid", "mjd"], kind="stable").reset_index(drop=True)

det_all.to_parquet(DET_PARQUET, index=False)
print(f"wrote {DET_PARQUET}  ({len(det_all):,} detection rows, "
      f"{det_all['oid'].nunique():,} objects)")
det_all.head()

wrote data\rq3b_ztf_lc\rq3b_ztf_detections.parquet  (1,303,797 detection rows, 11,826 objects)


,oid,mjd,fid,magpsf,sigmapsf,isdiffpos,isdiffpos_sign
0,ZTF17aaaaaal,58365.475035,1,20.125000,0.330419,-1,-1
1,ZTF17aaaaaal,58367.465868,1,18.851116,0.164315,1,1
2,ZTF17aaaaaal,58372.454236,1,19.148970,0.154458,1,1
3,ZTF17aaaaaal,58429.424155,1,18.802189,0.205484,1,1
4,ZTF17aaaaaal,58434.407963,1,18.905378,0.190114,1,1


In [7]:
oids_with_photometry = set(det_all["oid"].unique())
no_photometry = sorted(GOLD_OID_SET - oids_with_photometry) if not QUICK else \
                sorted(set(GOLD_OIDS) - oids_with_photometry)

band_counts = (det_all["fid"].value_counts(dropna=False)
               .rename_axis("fid").astype(int).to_dict())

manifest = {
    "created":               RUN_DATE,
    "created_utc":           datetime.datetime.now(datetime.timezone.utc).isoformat(),
    "source":                "ALeRCE query_detections(oid, format='pandas', survey='ztf')",
    "alerce_version":        getattr(_alerce_pkg, "__version__", "unknown"),
    "query_signature":       str(inspect.signature(alerce.query_detections)),
    "gold_objects":          len(GOLD_OID_SET),
    "objects_requested":     len(GOLD_OIDS),
    "objects_with_photometry": len(oids_with_photometry),
    "objects_no_photometry":   len(no_photometry),
    "objects_failed_transport": len(failed_oids),
    "detection_rows":        int(len(det_all)),
    "columns":               list(det_all.columns),
    "rows_per_band":         {str(k): int(v) for k, v in band_counts.items()},
    "median_detections_per_object": float(det_all.groupby("oid").size().median())
                                    if len(det_all) else 0.0,
    "fetch_minutes":         round(ELAPSED_S / 60, 2),
    "status_counts":         status_counts,
    "workers":               WORKERS,
    "max_retries":           MAX_RETRIES,
    "target_rate_per_min":   TARGET_RATE_PER_MIN,
    "throttle_events":       len(_throttle_events),
    "final_rate_per_min":    round(60.0 / LIMITER._interval, 1),
    "quick_mode":            QUICK,
    "no_photometry_oids":    no_photometry[:500],   # truncated sample for inspection
    "failed_oids":           failed_oids,
}
MANIFEST.write_text(json.dumps(manifest, indent=2))
print(json.dumps({k: v for k, v in manifest.items()
                  if k not in ("no_photometry_oids", "failed_oids")}, indent=2))

{
  "created": "2026-08-07",
  "created_utc": "2026-08-07T21:03:38.767929+00:00",
  "source": "ALeRCE query_detections(oid, format='pandas', survey='ztf')",
  "alerce_version": "2.3.1",
  "query_signature": "(oid: str | int, format: str = 'json', survey: str | None = None, index=None, sort=None)",
  "gold_objects": 11826,
  "objects_requested": 11826,
  "objects_with_photometry": 11826,
  "objects_no_photometry": 0,
  "objects_failed_transport": 0,
  "detection_rows": 1303797,
  "columns": [
    "oid",
    "mjd",
    "fid",
    "magpsf",
    "sigmapsf",
    "isdiffpos",
    "isdiffpos_sign"
  ],
  "rows_per_band": {
    "2": 696005,
    "1": 594463,
    "3": 13329
  },
  "median_detections_per_object": 34.0,
  "fetch_minutes": 92.71,
  "status_counts": {
    "cached": 3490,
    "fetched": 8336,
    "empty": 0,
    "failed": 0
  },
  "workers": 6,
  "max_retries": 4,
  "target_rate_per_min": 90.0,
  "throttle_events": 0,
  "final_rate_per_min": 90.0,
  "quick_mode": false
}


## 6. Closing assertions

The set of objects we obtained photometry for must be a **subset** of the gold object
set. A superset would mean this notebook had invented objects that no downstream chapter
knows about, and every join from here on would be silently wrong.

In [8]:
assert oids_with_photometry <= GOLD_OID_SET, (
    f"{len(oids_with_photometry - GOLD_OID_SET)} fetched oids are NOT in the gold set")
assert det_all["mjd"].notna().all(), "null MJD in stacked detections"
assert det_all["magpsf"].notna().all(), "null magpsf in stacked detections"

print(f"PASS  fetched oids are a subset of the gold oid set")
print(f"      gold objects              : {len(GOLD_OID_SET):,}")
print(f"      with photometry           : {len(oids_with_photometry):,}")
print(f"      WITHOUT any detections    : {len(no_photometry):,}")
print(f"      failed transport (retry)  : {len(failed_oids):,}")
print()
print(f"      detection rows            : {len(det_all):,}")
print(f"      g-band (fid=1) rows       : {band_counts.get(1, 0):,}")
print(f"      r-band (fid=2) rows       : {band_counts.get(2, 0):,}")

PASS  fetched oids are a subset of the gold oid set
      gold objects              : 11,826
      with photometry           : 11,826
      WITHOUT any detections    : 0
      failed transport (retry)  : 0

      detection rows            : 1,303,797
      g-band (fid=1) rows       : 594,463
      r-band (fid=2) rows       : 696,005
